In [ ]:
import pandas as pd
import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
import seaborn as sns 
from scipy.stats import pearsonr
import json
from shapely.geometry import shape 
# from shapely.geometry import Polygon 
import json 
from shapely import wkt 
from shapely.geometry import Point
from pandas.tseries.offsets import Week
from statsmodels.tsa.stattools import kpss 
import statsmodels.api as sm
import warnings
import scipy.stats as stats
from scipy.stats import zscore
import contextily as ctx
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from shapely.ops import unary_union
from scipy.spatial import cKDTree

In [ ]:
### Read hourly weather dataframe 
weather_df = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geo_Interpolation/Notebook/weather_concat.csv')

weather_df['Hour'] = pd.to_datetime(weather_df['Hour'])
weather_df = weather_df.rename(columns = {'Interpol_Precip':'Precip', 'Interpol_Temp':'Temp'})
weather_df = weather_df[['Hour', 'Precip', 'Temp', 'Location']]

In [ ]:
### Dictionary to store values of weather station names and data 

location_info = {
    'Location': ['Accra_Aca', 'G_Met', 'Temasco', 'St_Johns', 'Safisana', 'Nsawam', 'Agri_Impact', 'Accra_Girls', 'Legon', 'Madina', 'Berekuso'],
    'Coordinates': [
        (5.573103555, -0.244500082),
        (5.652019933, -0.16446233),
        (5.641413, -0.01187),
        (5.6383744, -0.2447151),
        (5.6836, -0.049468),
        (5.797283, -0.346325),
        (5.760172, -0.231223),
        (5.597071, -0.194248),
        (5.659788, -0.190434),
        (5.675314, -0.179166),
        (5.758029, -0.221716)
    ],
    'Elevation': [32.4, 71.6, 18.4, 46.0, 12.0, 62.0, 355.0, 63.0, 82.0, 60.4, 330.0]
}

In [ ]:
location_df = pd.DataFrame(location_info)

# Split coordinates into Latitude and Longitude
location_df[['Latitude', 'Longitude']] = pd.DataFrame(location_df['Coordinates'].tolist(), index=location_df.index)
location_df.drop(columns='Coordinates', inplace=True)

# Merge with weather_df
weather_df = weather_df.merge(location_df, on='Location', how='left')

weather_df['geometry'] = weather_df.apply(
    lambda row: Point(row['Longitude'], row['Latitude']), axis=1
)

weather_gdf = gpd.GeoDataFrame(weather_df, geometry='geometry')
weather_gdf = weather_gdf.set_crs(epsg=4326)
weather_gdf = weather_gdf.to_crs('EPSG:32630')

### Separate the weather data into different gdfs based on location/weather station 

In [ ]:
location_vars = {
    'Accra_Aca': 'accra_aca_df_year',
    'G_Met': 'g_met_hq_df_year',
    'Temasco': 'temasco_df_year',
    'St_Johns': 'st_johns_df_year',
    'Safisana': 'safisana_df_year',
    'Nsawam': 'nsawam_df_year',
    'Agri_Impact': 'agri_impact_df_year',
    'Accra_Girls': 'accra_girls_df_year',
    'Legon': 'legon_df_year',
    'Madina': 'madina_df_year',
    'Berekuso': 'berekuso_df_year'
}

# Loop through and create separate GeoDataFrames
for loc, var_name in location_vars.items():
    globals()[var_name] = weather_gdf[weather_gdf['Location'] == loc].copy()

## Interpolation of Weather Stations Workflow...

In [ ]:
# read 2022 shapefile of the weather station locations  
tahmo_test = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geo_Interpolation/Notebook/tahmo_gdf_2022.shp')

### Reproject Tahmo to EPSG 32630 
tahmo_test = tahmo_test.to_crs('EPSG:32630')
tahmo_test = tahmo_test.rename(columns = {'Site_ID':'Location'})

### Plot Weather Stations and Enumeration Areas 

In [ ]:
## read the Greater Accra shapefile data & reproject to local CRS 
ea_shapefile = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geo_Interpolation/Notebook/GAMA_20200923_eaBorder.shp')
ea_shapefile = ea_shapefile.to_crs('EPSG:32630')


## read shapefile of just the EAs with PQR data 
eas_of_interest = gpd.read_file("/home/kdonkor_umass_edu/Interpolation_Geospatial_All_EAs/study_area_copy.geojson")

In [ ]:
fig, ax = plt.subplots(figsize = (18,15), dpi = 250)

# plot of EAs without data 
ea_shapefile.plot(ax = ax, alpha = 0.5, edgecolor = 'white', lw = 0.6, color = 'grey', label = 'Enumeration Areas')

# EAs of Interest 
eas_of_interest.plot(ax = ax, alpha = 0.5, facecolor = 'yellow', lw = 0.6, label = 'EAs of Interest')

# Tahmo Stations Plot 
tahmo_test.plot(color = 'red', ax = ax, marker='*', markersize = 12, legend = True, label = 'Weather Station')

# basemap 
ctx.add_basemap(ax, zoom='auto', source=ctx.providers.OpenStreetMap.Mapnik, attribution=None, crs=(ea_shapefile.crs))

# Define legend entries  
enumeration_area_patch = mpatches.Patch(color='grey', alpha=0.5, label="Enumeration Area")
tahmo_patch = mlines.Line2D([], [], color='red', marker='*', linestyle='None', markersize=5, label="Weather Station")

# Add legend  
ax.legend(handles=[enumeration_area_patch, tahmo_patch], loc='lower right', fontsize=14, handleheight=2, handlelength=4)

# plot title 
ax.set_title('Study Area', fontsize = 18, pad = 10)

plt.show()

### Define radius around weather stations (10km --> lightning)

In [ ]:
buffer_radius = 10000

tahmo_buffers = tahmo_test.copy()
tahmo_buffers['geometry'] = tahmo_buffers.geometry.buffer(buffer_radius)

In [ ]:
# Create a figure and axis for plotting
fig, ax = plt.subplots(figsize=(18, 15), dpi=250)

# Plot Enumeration Areas (EA) without data
ea_shapefile.plot(ax=ax, alpha=0.5, edgecolor='white', lw=0.6, color='grey', label='Enumeration Areas')

# EAs of Interest 
eas_of_interest.plot(ax = ax, alpha = 0.5, facecolor = 'yellow', lw = 0.6, label = 'EAs of Interest')

# Plot Buffer circles (in semi-transparent color)
tahmo_buffers.plot(ax=ax, color='blue', alpha=0.3, label='10 km Buffer')

# Plot Tahmo Stations as red stars
tahmo_test.plot(color='red', ax=ax, marker='*', markersize=12, legend=True, label='Weather Station')

# Add Basemap
ctx.add_basemap(ax, zoom='auto', source=ctx.providers.OpenStreetMap.Mapnik, attribution=None, crs=(ea_shapefile.crs))

# Define legend entries  
enumeration_area_patch = mpatches.Patch(color='grey', alpha=0.5, label="Enumeration Area")
sensor_patch = mlines.Line2D([], [], color='red', marker='*', linestyle='None', markersize=5, label="Weather Station")
buffer_patch = mpatches.Patch(color='blue', alpha=0.3, label="Buffer")

# Add legend  
ax.legend(handles=[enumeration_area_patch, sensor_patch, buffer_patch], loc='lower right', fontsize=14, handleheight=2, handlelength=4)

# Plot title 
ax.set_title('Study Area', fontsize=18, pad=10)

# Show the plot
plt.show()

### Find the EAs that intersected with the weather station buffers 

In [ ]:
unioned_buffer = unary_union(tahmo_buffers['geometry'])

tahmo_buffer_union_gdf = gpd.GeoDataFrame(geometry=[unioned_buffer], crs=tahmo_buffers.crs)


## EAs that intersected with buffers 
eas_n_buffer = gpd.sjoin(eas_of_interest, tahmo_buffer_union_gdf, how = 'inner', predicate = 'intersects')

eas_n_buffer = eas_n_buffer.drop_duplicates(subset='ea_code9ch')
eas_n_buffer = eas_n_buffer[['ea_code9ch', 'geometry']]
eas_n_buffer = eas_n_buffer.reset_index(drop = True)

In [ ]:
## Plot to Visualize 


# Create a figure and axis for plotting
fig, ax = plt.subplots(figsize=(14, 10), dpi=250)

# Plot Enumeration Areas (EA) without data
ea_shapefile.plot(ax=ax, alpha=0.5, edgecolor='white', lw=0.6, color='grey', label='Enumeration Areas')

# EAs (INTERSECTING with Buffer) 
eas_n_buffer.plot(ax = ax, alpha = 0.5, facecolor = 'green', lw = 0.6, label = 'EAs Intersecting with Buffer')

# Plot Buffer circles (in semi-transparent color)
tahmo_buffers.plot(ax=ax, color='blue', alpha=0.3, label='5 km Buffer')

# Plot Tahmo Stations as red stars
tahmo_test.plot(color='red', ax=ax, marker='*', markersize=12, legend=True, label='Weather Station')

# Add Basemap
ctx.add_basemap(ax, zoom='auto', source=ctx.providers.OpenStreetMap.Mapnik, attribution=None, crs=(ea_shapefile.crs))

# Define legend entries  
enumeration_area_patch = mpatches.Patch(color='grey', alpha=0.5, label="Enumeration Area")
sensor_patch = mlines.Line2D([], [], color='red', marker='*', linestyle='None', markersize=5, label="Weather Station")
buffer_patch = mpatches.Patch(color='blue', alpha=0.3, label="Buffer")

# Add legend  
ax.legend(handles=[enumeration_area_patch, sensor_patch, buffer_patch], loc='lower right', fontsize=14, handleheight=2, handlelength=4)

ax.set_xlim(790000,834000)

# Plot title 
ax.set_title('Study Area', fontsize=18, pad=10)

# Show the plot
plt.show()

### Retrieve new list of pqr sites from these EAs 

In [ ]:
intersecting_sites_gdf = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Weather/Hourly_Weather_per_EA/intersecting_sites.geojson')

intersecting_sites_gdf = intersecting_sites_gdf[['space_grouping', 'ea_code9ch', 'geometry']]
intersecting_sites_gdf = intersecting_sites_gdf.rename(columns = {'space_grouping':'site_id'})
intersecting_sites_gdf['ea_code9ch'] = intersecting_sites_gdf['ea_code9ch'].astype(int)


new_list_eas = gpd.sjoin(intersecting_sites_gdf, eas_n_buffer, how = 'inner', predicate = 'intersects')
new_list_eas = new_list_eas[['site_id', 'geometry']].reset_index(drop=True)
new_list_eas = new_list_eas.drop_duplicates(subset='site_id')

new_pqr_site_list = np.sort(new_list_eas.site_id.unique().tolist())

### EAs and Weather Buffer matches 

In [ ]:
ea_with_station_matches = gpd.sjoin(eas_n_buffer, tahmo_buffers[['geometry', 'Location']], how = 'left', predicate = 'intersects')


ea_station_list = ea_with_station_matches.groupby(ea_with_station_matches.ea_code9ch)['Location'] \
                                         .apply(lambda x: list(set(x.dropna()))) \
                                         .reset_index(name='Intersecting_Stations')

### EAs and PQR site matches  

In [ ]:
ea_n_site_matches = gpd.sjoin(eas_n_buffer, intersecting_sites_gdf[['geometry', 'site_id']], how = 'left', predicate = 'intersects')


ea_site_list = ea_n_site_matches.groupby(ea_n_site_matches.ea_code9ch)['site_id'] \
                                         .apply(lambda x: list(set(x.dropna()))) \
                                         .reset_index(name='Intersecting_Sites')

### Combined EAs and PQRs 

In [ ]:
merged_eas_sites_stations = pd.merge(ea_site_list, ea_station_list, on = 'ea_code9ch', how = 'left')

### Weather Station --> Gdf Map 

In [ ]:
station_df_map = {
    'St_Johns': st_johns_df_year,
    'Temasco': temasco_df_year,
    'G_Met': g_met_hq_df_year,
    'Accra_Aca': accra_aca_df_year,
    'Safisana': safisana_df_year,
    'Nsawam': nsawam_df_year,
    'Agri_Impact': agri_impact_df_year,
    'Accra_Girls': accra_girls_df_year,
    'Legon': legon_df_year,
    'Madina': madina_df_year,
    'Berekuso': berekuso_df_year
}

### Assigning Lightning Events per EA 

In [ ]:
## read the hourly lightning data (all weather stations combined )
lightning_df = pd.read_csv('/home/kdonkor_umass_edu/Interpolation/combined_lightning_df.csv').drop(columns = ['Unnamed: 0'])

lightning_df['Timestamp'] = pd.to_datetime(lightning_df['Timestamp'])
lightning_df['Year'] = lightning_df['Timestamp'].dt.year  # Extract year from Timestamp

# Group by Year and Location, then sum
lightning_events_per_station_per_year = lightning_df.groupby(['Year', 'Location'])['Lightning Events'].sum().reset_index()
lightning_events_per_station_total = lightning_events_per_station_per_year.groupby('Location')['Lightning Events'].sum().reset_index()

In [ ]:
## Updated to ignore the hours with zero (0) lightning events (and uses other station dfs) 

def assign_avg_lightning_to_eas_ignore_zeros(lightning_df, ea_station_df):
    ea_results = []

    for _, row in ea_station_df.iterrows():
        ea_code = row['ea_code9ch']
        stations = row['Intersecting_Stations']

        # Filter for relevant stations
        matched = lightning_df[lightning_df['Location'].isin(stations)].copy()

        if not matched.empty:
            # Replace 0s with NaN for averaging so they're ignored
            matched['Lightning Events'] = matched['Lightning Events'].replace(0, np.nan)

            # Group by timestamp and average across stations, ignoring NaNs
            averaged = matched.groupby('Timestamp', as_index=False)['Lightning Events'].mean()

            # Fill any timestamps where all stations had 0s (now NaNs) with 0
            averaged['Lightning Events'] = averaged['Lightning Events'].fillna(0)

            # Round up to nearest integer
            averaged['Lightning Events'] = np.ceil(averaged['Lightning Events']).astype(int)

            averaged['ea_code9ch'] = ea_code
            ea_results.append(averaged)

    # Combine all EA results into one DataFrame
    final_df = pd.concat(ea_results, ignore_index=True)

    return final_df

#### Run Analysis - Assign Hourly Lightning Data to each EA  

In [ ]:
ea_hourly_lightning_avg = assign_avg_lightning_to_eas_ignore_zeros(lightning_df, ea_station_list)

#### Read in 289 EAs (6km buffer) 

In [ ]:
grid_weather_eas = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Weather/Hourly_Weather_per_EA/ea_code9ch_list_6km_buffer.csv')
list_eas = grid_weather_eas['ea_code9ch'].to_list()

#### Align EA-lightning data with 289 EAs 

In [ ]:
ea_hourly_lightning_avg = ea_hourly_lightning_avg[ea_hourly_lightning_avg['ea_code9ch'].isin(list_eas)].reset_index(drop=True)


ea_hourly_lightning_avg['Date'] = ea_hourly_lightning_avg['Timestamp'].dt.date
ea_hourly_lightning_avg['Month'] = ea_hourly_lightning_avg['Timestamp'].dt.month
ea_hourly_lightning_avg['Year'] = ea_hourly_lightning_avg['Timestamp'].dt.year

daily_lightning_per_ea = ea_hourly_lightning_avg.groupby(['ea_code9ch', 'Date'])['Lightning Events'].sum().reset_index()
monthly_lightning_per_ea = ea_hourly_lightning_avg.groupby(['ea_code9ch', 'Month'])['Lightning Events'].sum().reset_index()
yearly_lightning_per_ea = ea_hourly_lightning_avg.groupby(['ea_code9ch', 'Year'])['Lightning Events'].sum().reset_index()